In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Define paths
main_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset"
ct_folder = os.path.join(main_folder, "CT")
t1w_folder = os.path.join(main_folder, "MRI/T1W_DIR")
output_folder_train = "/content/drive/MyDrive/CT-MRI Paired Dataset/Model Train"

# Verify that Google Drive is mounted
if not os.path.exists(main_folder):
    raise FileNotFoundError(f"Main folder {main_folder} does not exist. Ensure Google Drive is mounted.")

# Create output folder if it doesn't exist
if not os.path.exists(output_folder_train):
    os.makedirs(output_folder_train)
    print(f"Created output folder: {output_folder_train}")

# Debug: Check if input folders exist
print(f"CT folder exists: {os.path.exists(ct_folder)}")
print(f"T1w folder exists: {os.path.exists(t1w_folder)}")

# Get sorted lists of files
ct_files = sorted([f for f in os.listdir(ct_folder) if f.endswith('_ct.nii') or f.endswith('_ct.nii.gz')])
t1w_files = sorted([f for f in os.listdir(t1w_folder) if f.endswith('_T1w.nii') or f.endswith('_T1w.nii.gz')])

# Debug: Print number of files found
print(f"Number of CT files: {len(ct_files)}")
print(f"Number of T1w files: {len(t1w_files)}")
print(f"Sample CT files: {ct_files[:3]}")
print(f"Sample T1w files: {t1w_files[:3]}")

# Verify file counts
assert len(ct_files) == len(t1w_files), f"Mismatch between CT ({len(ct_files)}) and T1w ({len(t1w_files)}) files"

def nifti_to_tensor(image_path):
    image = nib.load(image_path).get_fdata()
    num_slices = image.shape[2]
    center_slice = num_slices // 2
    tensor = image[:, :, center_slice]
    tensor = (tensor - np.min(tensor)) / (np.max(tensor) - np.min(tensor) + 1e-10)  # Avoid division by zero
    tensor = tf.image.resize(tf.expand_dims(tensor, -1), [256, 256], method='bilinear')
    tensor = tf.squeeze(tensor, -1).numpy()
    return tensor

# Process files
for ct_file, t1w_file in zip(ct_files, t1w_files):
    subject_id = ct_file.split('_')[0]
    assert subject_id == t1w_file.split('_')[0], f"Subject ID mismatch: {ct_file}, {t1w_file}"

    ct_path = os.path.join(ct_folder, ct_file)
    t1w_path = os.path.join(t1w_folder, t1w_file)

    # Debug: Confirm file paths
    print(f"Processing CT: {ct_path}")
    print(f"Processing T1w: {t1w_path}")

    # Verify files exist
    if not os.path.exists(ct_path) or not os.path.exists(t1w_path):
        print(f"Warning: File not found - CT: {ct_path}, T1w: {t1w_path}")
        continue

    # Convert to tensors
    ct_tensor = nifti_to_tensor(ct_path)
    t1w_tensor = nifti_to_tensor(t1w_path)

    ct_tensor = np.squeeze(ct_tensor)
    t1w_tensor = np.squeeze(t1w_tensor)

    # Define output paths
    ct_output_path = os.path.join(output_folder_train, f"{subject_id}_input.png")
    t1w_output_path = os.path.join(output_folder_train, f"{subject_id}_target.png")

    # Save images
    plt.imsave(ct_output_path, ct_tensor, cmap='gray')
    plt.imsave(t1w_output_path, t1w_tensor, cmap='gray')

    # Debug: Confirm images were saved
    if os.path.exists(ct_output_path) and os.path.exists(t1w_output_path):
        print(f"Saved: {ct_output_path}")
        print(f"Saved: {t1w_output_path}")
    else:
        print(f"Failed to save: {ct_output_path} or {t1w_output_path}")

# Debug: List files in output folder
output_files = os.listdir(output_folder_train)
print(f"Files in output folder ({output_folder_train}): {len(output_files)}")
print(f"Sample output files: {output_files[:3]}")

print("All input and target images (center slice, resized to 256x256) have been successfully saved to the output folder.")

In [ ]:
"""
import nibabel as nib
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
import os

def convert_ct_to_tensor(input_path, target_size=(256, 256), output_dir="/kaggle/working/", tensor_filename="ct_tensor"):
"""
"""
    Convert a NIfTI CT image to a tensor, resize to target size, and save in output_dir.

    Args:
        input_path (str): Path to a NIfTI file.
        target_size (tuple): Target size for resizing (height, width).
        output_dir (str): Directory to save the tensor (.npy file).
        tensor_filename (str): Base filename for the saved tensor.

    Returns:
        np.ndarray: 2D tensor (target_size[0] × target_size[1]) for model input.
        np.ndarray: Original tensor (2D or 3D) before resizing.
"""
"""
    input_path = Path(input_path)
    if not input_path.is_file() or input_path.suffix not in ['.nii', '.gz']:
        raise ValueError("Input path must be a .nii or .nii.gz file.")

    # Load NIfTI file
    nii_img = nib.load(input_path)
    pixel_array = np.array(nii_img.get_fdata())
    nii_img.uncache()

    # Handle 2D or 3D
    if pixel_array.ndim == 2:
        tensor = pixel_array
    elif pixel_array.ndim == 3:
        # Select middle slice for 3D images
        tensor = pixel_array[:, :, pixel_array.shape[2] // 2]
    else:
        raise ValueError(f"Unexpected pixel array shape: {pixel_array.shape}")

    # Save original tensor
    os.makedirs(output_dir, exist_ok=True)
    base_name = input_path.stem
    tensor_path = os.path.join(output_dir, f"{tensor_filename}_{base_name}.npy")
    np.save(tensor_path, pixel_array)
    print(f"Saved tensor to {tensor_path}")

    # Resize for model input
    tensor_resized = tf.expand_dims(tensor, axis=-1)  # Add channel dimension
    tensor_resized = tf.image.resize(tensor_resized, target_size, method='bilinear')
    tensor_resized = tf.squeeze(tensor_resized, axis=-1).numpy()  # Remove channel dimension

    return tensor_resized, pixel_array

# Process CT and T1w images
ct_dir = "/kaggle/input/ct-mri-paired-dataset/CT"
t1w_dir = "/kaggle/input/ct-mri-paired-dataset/MRI/T1W_DIR"
output_dir = "/kaggle/working/model_train/"
os.makedirs(output_dir, exist_ok=True)

ct_files = sorted([f for f in Path(ct_dir).glob("*.nii")])
t1w_files = sorted([f for f in Path(t1w_dir).glob("*.nii")])

if len(ct_files) != len(t1w_files):
    raise ValueError("Number of CT and T1w files must match.")

for i, (ct_file, t1w_file) in enumerate(zip(ct_files, t1w_files)):
    # Convert CT and T1w to tensors
    ct_tensor_resized, ct_tensor_original = convert_ct_to_tensor(
        ct_file, output_dir=output_dir, tensor_filename=f"ct_tensor_{i}"
    )
    t1w_tensor_resized, t1w_tensor_original = convert_ct_to_tensor(
        t1w_file, output_dir=output_dir, tensor_filename=f"t1w_tensor_{i}"
    )

    # Save resized tensors as PNGs for model input
    plt.imsave(os.path.join(output_dir, f"{i}_input.png"), ct_tensor_resized, cmap='gray')
    plt.imsave(os.path.join(output_dir, f"{i}_target.png"), t1w_tensor_resized, cmap='gray')

print(f"Processed {len(ct_files)} CT and T1w pairs.")
"""

In [ ]:
print(ct_tensor.shape)
print(t1w_tensor.shape)

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Define paths
main_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset"
ct_folder = os.path.join(main_folder, "CT")
t1w_folder = os.path.join(main_folder, "MRI/T1W_DIR")
output_folder_test = "/content/drive/MyDrive/CT-MRI Paired Dataset/Model Test"

# Verify that Google Drive is mounted
if not os.path.exists(main_folder):
    raise FileNotFoundError(f"Main folder {main_folder} does not exist. Ensure Google Drive is mounted.")

# Create output folder if it doesn't exist
if not os.path.exists(output_folder_test):
    os.makedirs(output_folder_test)
    print(f"Created output folder: {output_folder_test}")

# Debug: Check if input folders exist
print(f"CT folder exists: {os.path.exists(ct_folder)}")
print(f"T1w folder exists: {os.path.exists(t1w_folder)}")

# Get sorted lists of files, supporting both .nii and .nii.gz
ct_files = sorted([f for f in os.listdir(ct_folder) if f.endswith('_ct.nii') or f.endswith('_ct.nii.gz')])
t1w_files = sorted([f for f in os.listdir(t1w_folder) if f.endswith('_T1w.nii') or f.endswith('_T1w.nii.gz')])

# Debug: Print number of files found
print(f"Number of CT files: {len(ct_files)}")
print(f"Number of T1w files: {len(t1w_files)}")
print(f"Sample CT files: {ct_files[:3]}")
print(f"Sample T1w files: {t1w_files[:3]}")

# Verify file counts
assert len(ct_files) == len(t1w_files), f"Mismatch between CT ({len(ct_files)}) and T1w ({len(t1w_files)}) files"

def nifti_to_tensor(image_path):
    image = nib.load(image_path).get_fdata()
    num_slices = image.shape[2]
    center_slice = num_slices // 2
    tensor = image[:, :, center_slice]
    tensor = (tensor - np.min(tensor)) / (np.max(tensor) - np.min(tensor) + 1e-10)  # Avoid division by zero
    tensor = tf.image.resize(tf.expand_dims(tensor, -1), [256, 256], method='bilinear')
    tensor = tf.squeeze(tensor, -1).numpy()
    return tensor

# Process files
for ct_file, t1w_file in zip(ct_files, t1w_files):
    subject_id = ct_file.split('_')[0]
    assert subject_id == t1w_file.split('_')[0], f"Subject ID mismatch: {ct_file}, {t1w_file}"

    ct_path = os.path.join(ct_folder, ct_file)
    t1w_path = os.path.join(t1w_folder, t1w_file)

    # Debug: Confirm file paths
    print(f"Processing CT: {ct_path}")
    print(f"Processing T1w: {t1w_path}")

    # Verify files exist
    if not os.path.exists(ct_path) or not os.path.exists(t1w_path):
        print(f"Warning: File not found - CT: {ct_path}, T1w: {t1w_path}")
        continue

    # Convert to tensors
    ct_tensor = nifti_to_tensor(ct_path)
    t1w_tensor = nifti_to_tensor(t1w_path)

    ct_tensor = np.squeeze(ct_tensor)
    t1w_tensor = np.squeeze(t1w_tensor)

    # Define output paths
    ct_output_path = os.path.join(output_folder_test, f"{subject_id}_input.png")
    t1w_output_path = os.path.join(output_folder_test, f"{subject_id}_target.png")

    # Save images
    plt.imsave(ct_output_path, ct_tensor, cmap='gray')
    plt.imsave(t1w_output_path, t1w_tensor, cmap='gray')

    # Debug: Confirm images were saved
    if os.path.exists(ct_output_path) and os.path.exists(t1w_output_path):
        print(f"Saved: {ct_output_path}")
        print(f"Saved: {t1w_output_path}")
    else:
        print(f"Failed to save: {ct_output_path} or {t1w_output_path}")

# Debug: List files in output folder
output_files = os.listdir(output_folder_test)
print(f"Files in output folder ({output_folder_test}): {len(output_files)}")
print(f"Sample output files: {output_files[:3]}")

print("All input and target images (center slice, resized to 256x256) have been successfully saved to the output folder.")

In [ ]:
# import tensorflow as tf
# BUFFER_SIZE = 52
# BATCH_SIZE = 1
# # Get lists of T1, FLAIR, and PET scan file paths
# input_files = sorted(tf.io.gfile.glob(os.path.join('/content/drive/MyDrive/MNIold/model_train', '*_input.png')))
# target_files = sorted(tf.io.gfile.glob(os.path.join('/content/drive/MyDrive/MNIold/model_train', '*_target.png')))

# # Make sure the lists of T1, FLAIR, and PET scan file paths have the same length
# assert len(input_files) == len(target_files)

# # Create a dataset of tuples, where each tuple is (T1 path, FLAIR path, PET path) for a single patient
# file_paths_dataset_train = tf.data.Dataset.from_tensor_slices((input_files, target_files))

# # Shuffle the dataset
# file_paths_dataset_train = file_paths_dataset_train.shuffle(BUFFER_SIZE)

# # Batch the dataset
# file_paths_dataset_train = file_paths_dataset_train.batch(BATCH_SIZE)


In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices('GPU'))
BUFFER_SIZE = 52
BATCH_SIZE = 1

input_files = sorted(tf.io.gfile.glob(os.path.join('/content/drive/MyDrive/CT-MRI Paired Dataset/Model Train', '*_input.png')))
target_files = sorted(tf.io.gfile.glob(os.path.join('/content/drive/MyDrive/CT-MRI Paired Dataset/Model Train', '*_target.png')))
assert len(input_files) == len(target_files)
file_paths_dataset_train = tf.data.Dataset.from_tensor_slices((input_files, target_files))

def load_and_preprocess_images(input_path, target_path):
    input_image = tf.io.read_file(input_path)
    target_image = tf.io.read_file(target_path)
    input_image = tf.image.decode_png(input_image, channels=1)
    target_image = tf.image.decode_png(target_image, channels=1)
    input_image = tf.image.convert_image_dtype(input_image, tf.float32)
    target_image = tf.image.convert_image_dtype(target_image, tf.float32)

    # Ensure images are [256, 256]
    input_image = tf.image.resize(input_image, [256, 256], method='bilinear')
    target_image = tf.image.resize(target_image, [256, 256], method='bilinear')
    return input_image, target_image

file_paths_dataset_train = file_paths_dataset_train.map(load_and_preprocess_images)
file_paths_dataset_train = file_paths_dataset_train.shuffle(BUFFER_SIZE)
file_paths_dataset_train = file_paths_dataset_train.batch(BATCH_SIZE)
file_paths_dataset_train = file_paths_dataset_train.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

In [ ]:
input_files_test = sorted(tf.io.gfile.glob(os.path.join('/content/drive/MyDrive/CT-MRI Paired Dataset/Model Test', '*_input.png')))
target_files_test = sorted(tf.io.gfile.glob(os.path.join('/content/drive/MyDrive/CT-MRI Paired Dataset/Model Test', '*_target.png')))

assert len(input_files_test) == len(target_files_test)

file_paths_dataset_test = tf.data.Dataset.from_tensor_slices((input_files_test, target_files_test))

file_paths_dataset_test = file_paths_dataset_test.map(load_and_preprocess_images)
file_paths_dataset_test = file_paths_dataset_test.batch(BATCH_SIZE)
file_paths_dataset_test = file_paths_dataset_test.prefetch(buffer_size=tf.data.experimental.AUTOTUNE)

In [ ]:
def downsample(filters, size, apply_batchnorm=True):
  initializer = tf.random_normal_initializer(0., 0.02)

  result = tf.keras.Sequential()
  result.add(
      tf.keras.layers.Conv2D(filters, size, strides=2, padding='same',
                             kernel_initializer=initializer, use_bias=False))

  if apply_batchnorm:
    result.add(tf.keras.layers.BatchNormalization())

  result.add(tf.keras.layers.LeakyReLU())

  return result

In [ ]:
# Assuming OUTPUT_CHANNELS is set to 1 (for grayscale images)
OUTPUT_CHANNELS = 1

def upsample(filters, size, apply_dropout=False):
  initializer = tf.random_normal_initializer(0., 0.02)

  result = tf.keras.Sequential()
  result.add(
    tf.keras.layers.Conv2DTranspose(filters, size, strides=2,
                                    padding='same',
                                    kernel_initializer=initializer,
                                    use_bias=False))

  result.add(tf.keras.layers.BatchNormalization())

  if apply_dropout:
      result.add(tf.keras.layers.Dropout(0.5))

  result.add(tf.keras.layers.ReLU())

  return result

In [ ]:
print(ct_tensor.shape)

In [ ]:
# Add an extra dimension to represent the batch size and channels
ct_tensor = tf.expand_dims(ct_tensor, 0)  # Add batch size dimension
ct_tensor = tf.expand_dims(ct_tensor, -1)  # Add channel dimension

# Define the number of filters and the size of the filters
filters = 64
filter_size = 4

# Create the downsampling model
down_model = downsample(filters, filter_size)

# Now apply the downsampling model to the input image
down_result = down_model(ct_tensor)

print(down_result.shape)

In [ ]:
up_model = upsample(1, 4)
up_result = up_model(down_result)
up_result = up_result[:, :256, :256, :]  # Updated to 256x256
print(up_result.shape)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Concatenate, Conv2DTranspose, Conv2D, Lambda, Cropping2D
from keras.saving import register_keras_serializable

@register_keras_serializable()
def resize_function(x):
    return tf.image.resize(x, [256, 256])

def Generator():
    inputs = Input(shape=[256, 256, 1])  # Input shape for 256x256 images

    down_stack = [
        downsample(64, 4, apply_batchnorm=False),  # (batch_size, 128, 128, 64)
        downsample(128, 4),  # (batch_size, 64, 64, 128)
        downsample(256, 4),  # (batch_size, 32, 32, 256)
        downsample(512, 4),  # (batch_size, 16, 16, 512)
        downsample(512, 4),  # (batch_size, 8, 8, 512)
        downsample(512, 4),  # (batch_size, 4, 4, 512)
        downsample(512, 4),  # (batch_size, 2, 2, 512)
    ]

    up_stack = [
        upsample(512, 4, apply_dropout=True),  # (batch_size, 4, 4, 1024)
        upsample(512, 4, apply_dropout=True),  # (batch_size, 8, 8, 1024)
        upsample(512, 4, apply_dropout=True),  # (batch_size, 16, 16, 1024)
        upsample(512, 4),  # (batch_size, 32, 32, 1024)
        upsample(256, 4),  # (batch_size, 64, 64, 512)
        upsample(128, 4),  # (batch_size, 128, 128, 256)
        upsample(64, 4),   # (batch_size, 256, 256, 128)
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = Conv2DTranspose(1, 4, strides=2, padding='same',
                           kernel_initializer=initializer,
                           activation='tanh')  # (batch_size, 512, 512, 1)
    x = inputs
    # Downsampling through the model
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)
    skips = reversed(skips[:-1])

    # Upsampling and establishing the skip connections
    for up, skip in zip(up_stack, skips):
        x = up(x)
        # Calculate cropping to match skip connection shape
        skip_h, skip_w = skip.shape[1], skip.shape[2]
        x_h, x_w = x.shape[1], x.shape[2]
        crop_h = (x_h - skip_h) // 2
        crop_w = (x_w - skip_w) // 2
        x = Cropping2D(cropping=((crop_h, crop_h), (crop_w, crop_w)))(x)
        x = Concatenate()([x, skip])
    x = last(x)
    final = Conv2D(1, 1, strides=1, padding='same',
                   kernel_initializer=initializer,
                   activation='tanh')  # (batch_size, 512, 512, 1)
    x = final(x)

    # Use registered function for Lambda layer
    x = Lambda(
        resize_function,
        output_shape=(256, 256, 1)
    )(x)
    return tf.keras.Model(inputs=inputs, outputs=x)

In [ ]:
generator = Generator()
tf.keras.utils.plot_model(generator, show_shapes=True, dpi=64)

In [ ]:
# Instantiate the generator model
generator = Generator()

# Resize ct_tensor to [256, 256, 1]
ct_tensor_resized = tf.image.resize(ct_tensor, [256, 256], method='bilinear')

# Get the output of the generator
output = generator(ct_tensor_resized)

# The output is a 4D tensor (batch_size, height, width, channels)
# We can visualize the first image in the batch
gen_output = output[0, :, :, 0]

# Use matplotlib to visualize the image
import matplotlib.pyplot as plt
plt.imshow(gen_output, cmap='gray')
plt.show()

In [ ]:
# import matplotlib.pyplot as plt

# # Assume that 'output_tensor' is the output from your generator
# # Remove batch dimension and convert tensor to numpy array for plotting
# output_image = output_tensor[0].numpy()

# # Since the output image is in the range [-1, 1], we bring it back to the range [0, 1]
# # output_image = (output_image + 1) / 2.0

# # Plot the output image
# plt.imshow(output_image, cmap='gray')
# plt.show()


In [ ]:
LAMBDA = 100
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(disc_generated_output, gen_output, target):
    # GAN loss
    gan_loss = loss_object(tf.ones_like(disc_generated_output), disc_generated_output)

    # Mean absolute error (L1 loss)
    l1_loss = tf.reduce_mean(tf.abs(target - gen_output))

    # Total generator loss
    total_gen_loss = gan_loss + (LAMBDA * l1_loss)

    return total_gen_loss, gan_loss, l1_loss

In [ ]:
def Discriminator():
    initializer = tf.random_normal_initializer(0., 0.02)

    inp = tf.keras.layers.Input(shape=[256, 256, 1], name='input_image')
    tar = tf.keras.layers.Input(shape=[256, 256, 1], name='target_image')

    x = tf.keras.layers.concatenate([inp, tar])  # (batch_size, 256, 256, 2)

    down1 = downsample(64, 4, False)(x)  # (batch_size, 128, 128, 64)
    down2 = downsample(128, 4)(down1)  # (batch_size, 64, 64, 128)
    down3 = downsample(256, 4)(down2)  # (batch_size, 32, 32, 256)

    zero_pad1 = tf.keras.layers.ZeroPadding2D()(down3)  # (batch_size, 34, 34, 256)
    conv = tf.keras.layers.Conv2D(512, 4, strides=1,
                                  kernel_initializer=initializer,
                                  use_bias=False)(zero_pad1)  # (batch_size, 31, 31, 512)

    batchnorm1 = tf.keras.layers.BatchNormalization()(conv)

    leaky_relu = tf.keras.layers.LeakyReLU()(batchnorm1)

    zero_pad2 = tf.keras.layers.ZeroPadding2D()(leaky_relu)  # (batch_size, 33, 33, 512)

    last = tf.keras.layers.Conv2D(1, 4, strides=1,
                                  kernel_initializer=initializer)(zero_pad2)  # (batch_size, 30, 30, 1)

    return tf.keras.Model(inputs=[inp, tar], outputs=last)

In [ ]:
discriminator = Discriminator()
tf.keras.utils.plot_model(discriminator, show_shapes=True, dpi=64)

In [ ]:
# Print the shape of the output tensor
print(ct_tensor.shape)  # Should be [1, 207, 243, 1] from Cell 12

# Expand t1w_tensor to 4D (batch_size, height, width, channels)
t1w_tensor = tf.expand_dims(t1w_tensor, axis=-1)  # Add channel dimension
t1w_tensor = tf.expand_dims(t1w_tensor, axis=0)  # Add batch dimension

# Expand gen_output from Block 3 to 4D
gen_output = tf.expand_dims(gen_output, axis=-1)  # Add channel dimension
gen_output = tf.expand_dims(gen_output, axis=0)  # Add batch dimension

print(t1w_tensor.shape)  # Should be [1, 207, 243, 1]
print(gen_output.shape)  # Should be [1, 207, 243, 1]

In [ ]:
import matplotlib.pyplot as plt

# Assume 'generator' and 'discriminator' are your defined models
# Assume 'ct_tensor' and 't1w_tensor' are your input data

# Generate an image with your generator (already done in Block 3)
# disc_out = discriminator([ct_tensor, t1w_tensor], training=False)  # Corrected comment

# Pass the generated image and the input image through the discriminator
disc_out = discriminator([ct_tensor, gen_output], training=False)

# The discriminator output is a 4D tensor: (batch_size, height, width, channels)
# We can take the mean over the batch and channels to get a 2D array
disc_out_mean = tf.reduce_mean(disc_out, axis=[0, 3])

# Plot the output
plt.imshow(disc_out_mean, cmap='gray')
plt.colorbar()
plt.show()

In [ ]:
loss_object = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def discriminator_loss(disc_real_output, disc_generated_output):
    real_loss = loss_object(tf.ones_like(disc_real_output), disc_real_output)

    generated_loss = loss_object(tf.zeros_like(disc_generated_output), disc_generated_output)

    total_disc_loss = real_loss + generated_loss

    return total_disc_loss

# Define the optimizers
generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# Instantiate the generator and the discriminator
generator = Generator()
discriminator = Discriminator()

def count_parameters(layer):
    num_params = 0
    for weight in layer.trainable_weights:
        num_params += tf.reduce_prod(weight.shape)
    return num_params

# Count parameters
total_gen_params = sum(count_parameters(layer) for layer in generator.layers)
total_disc_params = sum(count_parameters(layer) for layer in discriminator.layers)

print(f"Total generator parameters: {total_gen_params}")
print(f"Total discriminator parameters: {total_disc_params}")

In [ ]:
def generate_images(model, test_input, tar):
    # Generate a prediction from the model
    prediction = model(test_input, training=True)

    # Create a new figure
    plt.figure(figsize=(15, 15))

    # Create a list of the input, target, and prediction images
    display_list = [test_input[0], tar[0], prediction[0]]
    title = ['Input Image', 'Ground Truth', 'Predicted Image']

    # Loop over each image in the display list
    for i in range(3):
        # Create a subplot for each image
        plt.subplot(1, 3, i+1)
        plt.title(title[i])
        # Display the image
        # If your images are not in the range [0, 1], you might need to adjust this line
        plt.imshow(display_list[i] * 0.5 + 0.5, cmap='gray')
        plt.axis('off')
    plt.show()

In [ ]:
from IPython.display import clear_output
import time

train_losses = []

@tf.function
def train_step(input_image, target, step):
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        gen_output = generator(input_image, training=True)

        disc_real_output = discriminator([input_image, target], training=True)
        disc_generated_output = discriminator([input_image, gen_output], training=True)

        gen_total_loss, gen_gan_loss, gen_l1_loss = generator_loss(disc_generated_output, gen_output, target)
        disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

        train_losses.append(gen_total_loss)

    generator_gradients = gen_tape.gradient(gen_total_loss, generator.trainable_variables)
    discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

In [ ]:
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output
import tensorflow as tf

def fit(train_ds, steps, test_ds):
    example_input, example_target = next(iter(test_ds.take(1)))
    start = time.time()

    # Initialize empty lists to store losses
    generator_losses = []
    discriminator_losses = []
    training_mse_values = []  # To store training MSE
    training_mae_values = []  # To store training MAE

    for step, (input_image, target) in enumerate(train_ds.repeat().take(steps)):
        # Every 20 steps: clear the display and run generate_images to show the progress
        if step % 20 == 0:
            if step != 0:
                print(f'\nTime taken for 20 steps: {time.time() - start:.2f} sec')
            start = time.time()

            # Clear the previous output images
            clear_output(wait=True)

            # Assuming that generate_images is a function you've defined to display images
            generate_images(generator, example_input, example_target)
            print(f"\nStep: {step // 20}")

            # Calculate MSE and MAE on training dataset
            gen_output = generator(input_image, training=False)
            mse = tf.keras.losses.MeanSquaredError()
            mae = tf.keras.losses.MeanAbsoluteError()
            training_mse = mse(target, gen_output).numpy()
            training_mae = mae(target, gen_output).numpy()
            training_mse_values.append(training_mse)
            training_mae_values.append(training_mae)

            # Print training MSE and MAE
            print(f"Training MSE: {training_mse:.4f}")
            print(f"Training MAE: {training_mae:.4f}")

        # Training step
        with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
            gen_output = generator(input_image, training=True)

            disc_real_output = discriminator([input_image, target], training=True)
            disc_generated_output = discriminator([input_image, gen_output], training=True)

            gen_loss, _, _ = generator_loss(disc_generated_output, gen_output, target)
            disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

        # Update generator and discriminator weights
        generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
        discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
        generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
        discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

        # Append losses to lists
        generator_losses.append(gen_loss.numpy())
        discriminator_losses.append(disc_loss.numpy())

        # Every 10 steps print a dot (.)
        if (step + 1) % 10 == 0:
            print('.', end='', flush=True)

    # Plot final learning curves
    plot_learning_curves(generator_losses, discriminator_losses)

def plot_learning_curves(generator_losses, discriminator_losses):
    plt.figure(figsize=(10, 6))
    plt.plot(generator_losses, label='Generator Loss', color='blue')
    plt.plot(discriminator_losses, label='Discriminator Loss', color='red')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.title('GAN Learning Curves')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Call the modified fit function
fit(file_paths_dataset_train, 15000, file_paths_dataset_test)

In [ ]:
# import time
# import matplotlib.pyplot as plt
# from IPython.display import clear_output

# def fit(train_ds, steps, test_ds):
#     example_input, example_target = next(iter(test_ds.take(1)))
#     start = time.time()

#     # Initialize empty lists to store losses
#     generator_losses = []
#     discriminator_losses = []
#     training_mse_values = []  # To store training MSE
#     training_mae_values = []  # To store training MAE

#     for step, (input_image, target) in enumerate(train_ds.repeat().take(steps)):
#         # Every 20 steps: clear the display and run generate_images to show the progress
#         if step % 20 == 0:
#             if step != 0:
#                 print(f'\nTime taken for 20 steps: {time.time() - start:.2f} sec')
#             start = time.time()

#             # Clear the previous output images
#             clear_output(wait=True)

#             # Assuming that generate_images is a function you've defined to display images
#             generate_images(generator, example_input, example_target)
#             print(f"\nStep: {step // 20}")

#         # Training step
#         with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
#             gen_output = generator(input_image, training=True)

#             disc_real_output = discriminator([input_image, target], training=True)
#             disc_generated_output = discriminator([input_image, gen_output], training=True)

#             gen_loss, _, _ = generator_loss(disc_generated_output, gen_output, target)
#             disc_loss = discriminator_loss(disc_real_output, disc_generated_output)

#         # Update generator and discriminator weights
#         generator_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
#         discriminator_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
#         generator_optimizer.apply_gradients(zip(generator_gradients, generator.trainable_variables))
#         discriminator_optimizer.apply_gradients(zip(discriminator_gradients, discriminator.trainable_variables))

#         # Append losses to lists
#         generator_losses.append(gen_loss.numpy())
#         discriminator_losses.append(disc_loss.numpy())

#         # Every 10 steps print a dot (.)
#         if (step + 1) % 10 == 0:
#             print('.', end='', flush=True)

#     # Plot learning curves
#     plot_learning_curves(generator_losses, discriminator_losses)

# def plot_learning_curves(generator_losses, discriminator_losses):
#     plt.figure(figsize=(10, 6))
#     plt.plot(generator_losses, label='Generator Loss', color='blue')
#     plt.plot(discriminator_losses, label='Discriminator Loss', color='red')
#     plt.xlabel('Training Steps')
#     plt.ylabel('Loss')
#     plt.title('GAN Learning Curves')
#     plt.legend()
#     plt.grid(True)
#     plt.show()

# # Call the modified fit function
# fit(file_paths_dataset_train, 10000, file_paths_dataset_test)



In [ ]:
# from IPython.display import clear_output

# def fit(train_ds, steps, test_ds):
#     example_input, example_target = next(iter(test_ds.take(1)))
#     start = time.time()

#     for step, (input_image, target) in train_ds.repeat().take(steps).enumerate():
#         # Every 20 steps: clear the display and run generate_images to show the progress
#         if (step) % 20 == 0:
#             if step != 0:
#                 print(f'\nTime taken for 20 steps: {time.time()-start:.2f} sec')
#             start = time.time()

#             # Clear the previous output images
#             clear_output(wait=True)

#             # Assuming that generate_images is a function you've defined to display images
#             generate_images(generator, example_input, example_target)
#             print(f"\nStep: {step//20}")

#         # Training step
#         train_step(input_image, target, step.numpy())

#         # Every 10 steps print a dot (.)
#         if (step+1) % 10 == 0:
#             print('.', end='', flush=True)


In [ ]:
train_losses = [tf.reduce_mean(loss).numpy() for loss in train_losses]

# Plot the learning curve
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label="Training Loss", marker='o')
plt.xlabel("Step")  # Changed from Epoch to Step
plt.ylabel("Loss")
plt.title("Learning Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model_folder = '/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models'
model_path = os.path.join(model_folder, "trainedpix2pix.h5")
generator.save(model_path)

In [ ]:
from tensorflow.keras.models import load_model

# Specify the path to your saved model
model_path = '/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/trainedpix2pix.h5'

# Load the model
loaded_model = load_model(model_path)

print("Model loaded successfully.")

In [ ]:
# Import necessary libraries
from keras.models import load_model
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import os

# Verify that the model file exists
model_path = '/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/trainedpix2pix.h5'
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model file not found at {model_path}. Ensure the model was trained and saved.")

# Load the model from the file
trained_generator = load_model(model_path)

# Print a success message
print("The trained generator model was successfully loaded from 'trainedpix2pix.h5'")

# Check if file_paths_dataset_test is defined
try:
    file_paths_dataset_test
except NameError:
    raise NameError("file_paths_dataset_test is not defined. Ensure the test dataset is created before running this code.")

# Initialize the error metrics
mse_total = 0
mae_total = 0
ssim_total = 0
psnr_total = 0
num_samples = 0

# Iterate over the test dataset
for input_image, target in file_paths_dataset_test:
    # Generate output image from the input image
    prediction = trained_generator(input_image, training=False)

    # Process each image in the batch
    for i in range(input_image.shape[0]):
        # Extract single images
        pred_img = prediction[i, :, :, 0].numpy()  # Shape: (256, 256)
        target_img = target[i, :, :, 0].numpy()    # Shape: (256, 256)

        # Rescale images from [-1, 1] to [0, 1] for SSIM and PSNR
        pred_img_rescaled = (pred_img + 1) / 2
        target_img_rescaled = (target_img + 1) / 2

        # Calculate MSE and MAE
        mse = tf.keras.losses.MeanSquaredError()
        mae = tf.keras.losses.MeanAbsoluteError()
        mse_total += mse(target_img, pred_img).numpy()
        mae_total += mae(target_img, pred_img).numpy()

        # Calculate SSIM
        ssim_value = ssim(target_img_rescaled, pred_img_rescaled, data_range=1.0)
        ssim_total += ssim_value

        # Calculate PSNR
        psnr_value = psnr(target_img_rescaled, pred_img_rescaled, data_range=1.0)
        psnr_total += psnr_value

        num_samples += 1

        # Display the input, target, and predicted images
        plt.figure(figsize=(15, 5))

        plt.subplot(1, 3, 1)
        plt.title('Input Image (CT)')
        plt.imshow(input_image[i, :, :, 0], cmap='gray')
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.title('Real Target Image (T1w)')
        plt.imshow(target_img, cmap='gray')
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.title('Predicted Image (T1w)')
        plt.imshow(pred_img, cmap='gray')
        plt.axis('off')

        plt.show()

# Calculate the average error metrics
mse_average = mse_total / num_samples
mae_average = mae_total / num_samples
ssim_average = ssim_total / num_samples
psnr_average = psnr_total / num_samples

print("Average Mean Squared Error:", mse_average)
print("Average Mean Absolute Error:", mae_average)
print("Average SSIM:", ssim_average)
print("Average PSNR (dB):", psnr_average)

In [ ]:
y_true = tf.reshape(target, [-1])  # Flatten the true labels
y_pred = tf.reshape(prediction, [-1])  # Flatten the predicted labels

# Accuracy calculation is invalid for regression; compute MSE instead
mse = tf.keras.losses.MeanSquaredError()
print("MSE:", mse(y_true, y_pred).numpy())

In [ ]:
import os
import matplotlib.pyplot as plt
import tensorflow as tf

# Define the path to the new folder where the images will be stored
new_output_folder = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/results_pix2PixTEST"

# Create the new folder if it doesn't exist
if not os.path.exists(new_output_folder):
    os.makedirs(new_output_folder)

# Iterate through the test dataset and save images for each subject
for idx, (input_image, target) in enumerate(file_paths_dataset_test):
    # Generate output image from the input image
    prediction = trained_generator(input_image, training=False)

    # Resize the prediction to (256, 256)
    prediction_resized = tf.image.resize(prediction, [256, 256])

    # Create a unique filename for each subject (starting from 1)
    subject_id = f"sub-{idx + 1:04d}"  # e.g., sub-0001

    plt.imsave(os.path.join(new_output_folder, f"{subject_id}_predicted.png"), prediction_resized[0, :, :, 0].numpy(), cmap='gray')

print(f"Saved images for {len(file_paths_dataset_test)} subjects to {new_output_folder}")

In [ ]:
print(prediction.shape)

In [ ]:
import h5py
import tensorflow as tf
import numpy as np
import os
from google.colab import drive

# Check if Drive is mounted; mount if not
drive_path = "/content/drive"
if not os.path.exists(drive_path):
    drive.mount(drive_path)
else:
    print("Drive already mounted at /content/drive")

# Define save path in Drive
save_path = "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models"
weights_dir = os.path.join(save_path, "weights_dump")
os.makedirs(weights_dir, exist_ok=True)

# Define the custom resize function
@tf.keras.utils.register_keras_serializable()
def resize_function(x):
    # Assuming the function resizes or reformats to (256, 256, 1)
    return tf.image.resize(x, [256, 256], method='bilinear')

# Load the model with custom_objects
try:
    model = tf.keras.models.load_model(
        "/content/drive/MyDrive/CT-MRI Paired Dataset/Saved Models/trainedpix2pix.h5",
        custom_objects={'Custom>resize_function': resize_function}
    )
except Exception as e:
    print(f"Error loading model: {e}")
    raise

# Save the architecture to Drive
architecture_path = os.path.join(save_path, "model_architecture.txt")
with open(architecture_path, "w") as f:
    model.summary(print_fn=lambda x: f.write(x + "\n"))
print(f"Saved model architecture at: {architecture_path}")

# Save each layer's weights as .npy and .txt files to Drive
for i, layer in enumerate(model.layers):
    layer_weights = layer.get_weights()
    if layer_weights:  # Skip layers with no weights
        for j, weight_array in enumerate(layer_weights):
            # Save as .npy
            npy_filename = os.path.join(weights_dir, f"layer_{i}_weight_{j}.npy")
            np.save(npy_filename, weight_array)
            print(f"Saved {npy_filename} - shape {weight_array.shape}")
            # Save as .txt
            txt_filename = os.path.join(weights_dir, f"layer_{i}_weight_{j}.txt")
            np.savetxt(txt_filename, weight_array.flatten(), fmt='%f')
            print(f"Saved {txt_filename} - shape {weight_array.shape}")